## 🏗️ LeetCode 1046: Last Stone Weight
---

<div style="padding: 15px; border-left: 8px solid #FF9800; background-color: #fff3e0; color: #e65100; border-radius: 4px;">
    <strong>The Core Insight:</strong> Each turn requires the two heaviest stones. A max-heap gives us the two largest values in O(log n) each. Python's heapq is a min-heap — negate all values to simulate a max-heap.
</div>

### 🛠️ The Mathematical Model

At each step: pop the two maximum elements, compute difference, push back if nonzero. A max-heap (negated in Python) ensures O(log n) access to the maximum at every step.

$$\text{Total turns} \leq n-1 \quad \text{Each turn: 2 pops + at most 1 push} \Rightarrow O(n \log n)$$

---

### 📋 Problem

You have stones with positive weights. Each turn: take the two heaviest stones y ≥ x. If y == x: both destroyed. If y != x: x destroyed, y becomes y - x. Return the weight of the last remaining stone, or 0 if none.

**Example 1:**
```
Input:  stones = [2,7,4,1,8,1]
Output: 1
Explanation: 8,7→1 | 4,2→2 | 2,1→1 | 1,1→0 | Last stone: 1
```

**Constraints:** 1 ≤ stones.length ≤ 30 | 1 ≤ stones[i] ≤ 1000

### 🧠 Choose Your Mental Model

<table style="width:100%; border-collapse: collapse;">
    <tr style="background-color: #f2f2f2; text-align: left;">
        <th style="padding: 12px; border: 1px solid #ddd;">Model</th>
        <th style="padding: 12px; border: 1px solid #ddd;">The "Story"</th>
        <th style="padding: 12px; border: 1px solid #ddd;">Mechanism</th>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Rock Crusher</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"The machine always grabs the two heaviest rocks. It smashes them. If one survived, it goes back in the pile. Repeat until one or zero rocks remain. I need the 'two heaviest' instantly each turn."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Max-heap: root always = heaviest. Two pops = two heaviest. One conditional push.</td>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Priority Processor</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"Tasks scheduled by priority. Each round: process the top two priority tasks. The higher-priority task reduces by the lower-priority task's amount and re-queues if it has remaining work."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Same structure — priority queue / max-heap with conditional re-insert</td>
    </tr>
</table>

---

### ⚠️ Performance Warning

<div style="padding: 10px; border: 1px solid #ffe58f; background-color: #fffbe6; border-radius: 4px;">
    <strong>Note:</strong> Sorting descending each turn is O(n log n) per turn, O(n² log n) total. Max-heap gives O(log n) per operation, O(n log n) total.
</div>

## 🐢 Approach 1: Brute Force — $O(n^2 \log n)$

In [ ]:
def lastStoneWeight_brute(stones):
    """
    Brute Force: Sort descending each turn
    Time: O(n^2 log n) | Space: O(1)
    """
    stones = stones[:]   # don't mutate input
    while len(stones) > 1:
        stones.sort(reverse=True)
        heavy, light = stones[0], stones[1]
        stones = stones[2:]
        if heavy != light:
            stones.append(heavy - light)
    return stones[0] if stones else 0


print(lastStoneWeight_brute([2, 7, 4, 1, 8, 1]))   # Expected: 1
print(lastStoneWeight_brute([1]))                    # Expected: 1

## 🔬 Complexity Analysis: $O(n^2 \log n)$ vs. $O(n \log n)$

<div style="padding: 15px; border-left: 8px solid #2196F3; background-color: #f0f7ff; color: #0d47a1; border-radius: 4px;">
    <strong>The Core Insight:</strong> We repeatedly need the two largest elements. A max-heap provides the largest element as root in O(1), and extracting it takes O(log n). Two pops per turn = O(log n) per turn. Over n turns total, O(n log n). No full sort needed each turn.
</div>

---

## 📉 Why Brute Force Fails: The $O(n^2 \log n)$ Trap

Sort each turn: O(n log n) per turn × up to n turns = O(n² log n). With n = 1,000 stones: ~10 million sort operations per run.

| Input Type | Brute Force Performance | Reason |
|------------|------------------------|--------|
| **30 stones** | ~27,000 sort ops | Sort each of 29 turns on shrinking list |
| **Large n** | n² log n | Sort cost per turn × n turns |

---

## 🚀 The Optimal Approach: $O(n \log n)$

Build a max-heap (negate all values for Python's min-heap). Each turn: pop the two maximum values in O(log n) each. If they differ, push the difference back. Continue until ≤ 1 stone remains.

### The Key Lifecycle Rule
1. **Negate all values** to simulate max-heap with Python's min-heap
2. **Pop twice** to get the two heaviest (negated → most negative = heaviest)
3. **Push difference** (if nonzero) back as negated value

---

## ✅ Mathematical Proof

Each turn removes at least one stone (both destroyed if equal, one destroyed if different). Starting from n stones, at most n-1 turns. Each turn: O(log n) heap operations. Total: O(n log n).

<div style="padding: 10px; border-left: 8px solid #4CAF50; background-color: #f1f8f4; color: #1b5e20; border-radius: 4px;">
    <strong>✅ Summary:</strong> Max-heap provides O(log n) access to the maximum — no sort needed each turn. The negation trick is standard Python idiom: negate on push, negate on pop.
</div>

## 🚀 Approach 2: Max-Heap (Negated) — $O(n \log n)$
---

Instead of sorting each turn, we use a **negated min-heap** as a max-heap.

As we iterate:
1. Build the max-heap: `[-s for s in stones]`, then `heapq.heapify(heap)`
2. While len(heap) > 1: pop two heaviest (negate back), compare, push difference if nonzero
3. Return `(-heap[0]) if heap else 0`

In [ ]:
import heapq

def lastStoneWeight(stones: list[int]) -> int:
    """
    Optimal: Max-Heap via negation
    Time: O(n log n) | Space: O(n) for the heap
    """
    heap = [-s for s in stones]   # negate for max-heap simulation
    heapq.heapify(heap)           # O(n) — builds heap in linear time

    while len(heap) > 1:
        heavy = -heapq.heappop(heap)   # largest (negate back)
        light = -heapq.heappop(heap)   # second largest

        if heavy != light:
            heapq.heappush(heap, -(heavy - light))   # push survivor (negated)

    return -heap[0] if heap else 0


print("Optimal:", lastStoneWeight([2, 7, 4, 1, 8, 1]))   # Expected: 1
print("Optimal:", lastStoneWeight([1]))                    # Expected: 1
print("Optimal:", lastStoneWeight([2, 2]))                 # Expected: 0

## 🎤 5 Interview Q&A

### Q1: How do you simulate a max-heap with Python's heapq?

**Answer:** Negate all values on push; negate again on pop. Python's heapq is a min-heap, so pushing `-x` means the "smallest" in the heap is `-max(original)`, i.e., the negated maximum. Popping returns the most negative value = negative of the original maximum. Negate it back to get the true maximum.

---

### Q2: What does `heapq.heapify` do and what's its time complexity?

**Answer:** `heapify` rearranges a list in-place into a valid min-heap in O(n) time — not O(n log n). It uses a bottom-up sift-down approach, starting from the middle of the array. The O(n) proof uses the fact that most elements are near leaves and need only small sift-down operations.

---

### Q3: What is the time complexity of the optimal approach and why?

**Answer:** O(n log n). `heapify` builds the initial heap in O(n). Each of at most n-1 turns does 2 pops + at most 1 push, each O(log n). Total: O(n) + O(n × log n) = O(n log n). Much better than the brute force O(n² log n).

---

### Q4: When would you use `heapreplace` instead of pop + push?

**Answer:** `heapreplace(heap, val)` is an atomic pop + push in O(log n) — more efficient than two separate operations when you know you'll always push after popping. In Last Stone Weight, we sometimes don't push (when heavy == light), so separate pop + conditional push is correct. Use `heapreplace` only when push is unconditional.

---

### Q5: What are the edge cases to watch for?

**Answer:** (1) Single stone — while loop never executes, returns that stone. (2) Two equal stones — both destroyed, heap empty, return 0. (3) All stones equal — they cancel pairwise; return 0 if even count, the stone value if odd count. (4) Already sorted — heapify handles any input order.

## 📚 Key Terminology

| Term | Definition |
|------|------------|
| **Max-Heap** | Complete binary tree where each parent ≥ its children — root is always the maximum |
| **Negation Trick** | Simulating a max-heap with Python's min-heap by negating all values — most negative = original maximum |
| **heapify** | O(n) in-place conversion of a list to a valid heap — more efficient than n individual pushes |
| **heappop** | Extract and return the minimum (or negated maximum) in O(log n) |
| **heappush** | Insert an element maintaining heap property in O(log n) |
| **heapreplace** | Atomic pop + push in O(log n) — use when you always push after popping |
| **Priority Queue** | Abstract data structure supporting insert + extract-max/min in O(log n) — heap is the standard implementation |
| **Smash and Survive** | The problem's turn mechanic: larger stone absorbs smaller, difference survives if nonzero |

## 💼 The Citi Narrative

**Context:** Citi's batch job scheduler used a priority queue to manage competing ETL jobs. Jobs had priority scores (1-1000); each scheduling cycle, the two highest-priority jobs were compared — if they conflicted for a shared resource, the higher-priority job ran and its priority was reduced by the lower-priority job's score, which was then re-queued.

**Scenario:** With 50 jobs competing for limited database connections, naive scheduling (sort all 50 each cycle) ran in O(50 log 50) per cycle × many cycles = slow. A max-heap of job priorities handled each scheduling decision in O(log 50).

**How this pattern applied:** The max-heap maintained the priority order efficiently. Each scheduling round = two heappops + conditional heappush — exactly the Last Stone Weight mechanics with "job priorities" as "stone weights."

**Impact:** Scheduler overhead dropped from O(n log n) sort per cycle to O(log n) per decision. For a scheduler running hundreds of cycles per minute with 50+ competing jobs, this mattered for scheduling latency and throughput — critical for ensuring high-priority capacity analysis jobs ran before lower-priority report generation.

In [ ]:
# -------------------------------------------------------
# PRACTICE: Given a list of numbers, repeatedly replace the
# two largest numbers with their sum until one number remains.
# Return that number. (Variant: instead of difference, use sum)
# -------------------------------------------------------

import heapq

def smashAndSum(nums):
    # Your solution here — max-heap approach
    pass


# Test
print(smashAndSum([3, 1, 4, 1, 5]))   # Expected: 14 (3+1+4+1+5 — all additions)

## 🎯 Summary: Key Takeaways

### The Pattern
**Max-Heap (Negated)** — Repeatedly access the maximum in O(log n) using Python's min-heap with negated values

### When to Use It
- ✅ Repeatedly need the maximum element from a changing collection
- ✅ Simulation problems where you process the largest items each turn
- ✅ Priority queues where max-priority items are processed first
- ❌ **Don't use when:** You need both min and max — use two separate heaps (median problem pattern)

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| Brute Force (sort each turn) | $O(n^2 \log n)$ | $O(1)$ |
| Optimal (Max-Heap) | $O(n \log n)$ | $O(n)$ |

### Interview Confidence Checklist
- [ ] Can explain the negation trick for max-heap in Python
- [ ] Can explain heapify and its O(n) complexity
- [ ] Can write the solution from memory in 3 minutes
- [ ] Can explain when to use heapreplace vs separate pop + push
- [ ] Can map to job scheduler or priority processor use case

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master Last Stone Weight and you've internalized the max-heap pattern — every "process the two largest" problem is solved the same way. 🚀